# Credit Card Consumption Prediction

A supervised learning project that predicts customer credit card consumption from available demographic and behavioral information.

## 1. Project Overview

This project combines customer consumption, behavioral, and demographic data to predict credit card consumption. It demonstrates a regression workflow from data inspection through model comparison and evaluation.

## 2. Business Problem

- **Problem type:** Supervised learning
- **Task:** Regression
- **Target:** `cc_cons`
- **Objective:** Predict customer credit card consumption from the available customer attributes.
- **Business relevance:** Understanding and estimating customer spending behavior can support analysis of customer needs and usage patterns. This project does not estimate credit risk.

## 3. Dataset Description

The repository expects three Excel files in `data/`. The descriptions below are based on the file names and observed column names; they do not assume undocumented business definitions.

| Dataset component | Purpose | Representative columns | Target |
|---|---|---|---|
| `CreditConsumptionData.xlsx` | Target and customer identifier | `ID`, `cc_cons` | `cc_cons` |
| `CustomerBehaviorData.xlsx` | Customer behavior and account-related attributes | `ID`, `cc_cons_apr`, `dc_cons_apr`, `credit_count_apr`, `investment_1`, `loan_enq`, `emi_active` | None |
| `CustomerDemographics.xlsx` | Customer demographic and channel attributes | `ID`, `account_type`, `gender`, `age`, `Income`, `region_code`, `NetBanking_Flag` | None |

The files are not committed to this repository. Reproduction requires authorized access to the source data.

## 4. Import Libraries

In [85]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler

In [87]:
# Find the repository or data directory from the current working directory.
repo_root = Path.cwd()
candidate_dirs = [repo_root, repo_root.parent, repo_root / "data", repo_root.parent / "data"]
data_dir = next(
    (
        directory
        for directory in candidate_dirs
        if directory.is_dir() and (directory / "CreditConsumptionData.xlsx").is_file()
    ),
    None,
)

if data_dir is None:
    raise FileNotFoundError("Add the Excel files to the repository's data/ folder before running this notebook.")

data_dir

WindowsPath('c:/Users/Aditya singh/Desktop/Analytix/ML_project/Credit_card_prediction/data')

## 5. Load Data

The source Excel files are read from the repository-relative `data/` directory. They are merged by `ID` in the following section, and the source data files are excluded from the public repository.

In [88]:
cust_consup = pd.read_excel(data_dir / "CreditConsumptionData.xlsx")
cust_behave = pd.read_excel(data_dir / "CustomerBehaviorData.xlsx")
cust_demog = pd.read_excel(data_dir / "CustomerDemographics.xlsx")

print("cust_consup shape:", cust_consup.shape)
print("cust_behave shape:", cust_behave.shape)
print("cust_demog shape:", cust_demog.shape)

cust_consup shape: (20000, 2)
cust_behave shape: (20000, 39)
cust_demog shape: (20000, 10)


## 6. Data Understanding

Inspect the source tables' shapes and data types before merging them.

In [89]:
cust_consup.info()
cust_behave.info()
cust_demog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ID       20000 non-null  int64  
 1   cc_cons  15000 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 312.6 KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 39 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ID                     20000 non-null  int64  
 1   cc_cons_apr            20000 non-null  float64
 2   dc_cons_apr            20000 non-null  float64
 3   cc_cons_may            19999 non-null  float64
 4   dc_cons_may            19999 non-null  float64
 5   cc_cons_jun            20000 non-null  float64
 6   dc_cons_jun            19999 non-null  float64
 7   cc_count_apr           19999 non-null  float64
 8   cc_count_may           20000 non-null  int64  
 9   cc

## 7. Merge and Categorical Encoding

Merge the three source tables on the existing customer ID, remove the original excluded columns, and one-hot encode object columns as in the original analysis.

In [90]:
cust_data = cust_consup.merge(cust_behave, how="inner", on="ID").merge(cust_demog, how="inner", on="ID")
cust_data = cust_data.drop(
    [
        "personal_loan_active",
        "vehicle_loan_active",
        "personal_loan_closed",
        "vehicle_loan_closed",
        "Tenure_with_Bank",
    ],
    axis=1,
    errors="ignore",
)

cat_cols = cust_data.select_dtypes(include="object").columns
if len(cat_cols) > 0:
    cust_data = pd.get_dummies(cust_data, columns=cat_cols, drop_first=True)

cust_data = cust_data.replace({True: 1, False: 0})
print("Merged dataset shape:", cust_data.shape)
print("Final number of columns:", len(cust_data.columns))

Merged dataset shape: (20000, 47)
Final number of columns: 47


C:\Users\Aditya singh\AppData\Local\Temp\ipykernel_21924\1642158195.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cust_data = cust_data.replace({True: 1, False: 0})


## 8. Train-Test Split

Rows with a known `cc_cons` target are used for training and evaluation. Rows with missing targets are kept separately for prediction.

In [91]:
actual_data = cust_data.loc[~cust_data["cc_cons"].isnull()].copy()
prediction_data = cust_data.loc[cust_data["cc_cons"].isnull()].copy()

X = actual_data.drop(["ID", "cc_cons"], axis=1)
y = actual_data["cc_cons"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=12,
)

## 9. Outlier Analysis and Treatment

The original workflow caps the selected numeric training and test features using IQR limits calculated from the training set.

In [92]:
outlier_cols = [
    "credit_count_may",
    "credit_count_jun",
    "investment_1",
    "investment_3",
    "emi_active",
    "max_credit_amount_jun",
    "max_credit_amount_apr",
]

for col in outlier_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    X_train[col] = X_train[col].clip(lower_limit, upper_limit)
    X_test[col] = X_test[col].clip(lower_limit, upper_limit)

In [93]:
print(X_train.columns.tolist())

['cc_cons_apr', 'dc_cons_apr', 'cc_cons_may', 'dc_cons_may', 'cc_cons_jun', 'dc_cons_jun', 'cc_count_apr', 'cc_count_may', 'cc_count_jun', 'dc_count_apr', 'dc_count_may', 'dc_count_jun', 'card_lim', 'investment_1', 'investment_2', 'investment_3', 'investment_4', 'debit_amount_apr', 'credit_amount_apr', 'debit_count_apr', 'credit_count_apr', 'max_credit_amount_apr', 'debit_amount_may', 'credit_amount_may', 'credit_count_may', 'debit_count_may', 'max_credit_amount_may', 'debit_amount_jun', 'credit_amount_jun', 'credit_count_jun', 'debit_count_jun', 'max_credit_amount_jun', 'emi_active', 'age', 'Emp_Tenure_Years', 'region_code', 'NetBanking_Flag', 'Avg_days_between_transaction', 'account_type_current', 'account_type_saving', 'gender_F', 'gender_M', 'Income_HIGH', 'Income_LOW', 'Income_MEDIUM']


## 10. Missing Value Treatment

The original code fills selected numeric columns with their training-set median and selected flag-like columns with their training-set mode. The fitted value from `X_train` is then used for both training and test data.

In [94]:
cols_median = [
    "cc_cons_may",
    "dc_cons_may",
    "investment_3",
    "dc_cons_jun",
    "cc_count_apr",
]

for col in cols_median:
    median_value = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

cols_mode = [
    "debit_count_apr",
    "emi_active",
]

for col in cols_mode:
    mode_value = X_train[col].mode()[0]
    X_train[col] = X_train[col].fillna(mode_value)
    X_test[col] = X_test[col].fillna(mode_value)

print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())

Missing values in X_train: 0
Missing values in X_test: 0


## 11. Feature Engineering

No derived features are introduced because the available column definitions and target time horizon are not documented well enough to justify additional transformations. The customer identifier and target are excluded from predictors; the observed customer attributes are otherwise retained.

## 12. Feature Scaling

The original workflow fits `StandardScaler` on the training features and applies that scaler to the test features.

In [95]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

Train rows: 12000
Test rows: 3000


## 13. Model Building

The original analysis evaluates Linear Regression, K-Nearest Neighbors, and Decision Tree regressors before tuning the existing tree and KNN models. LightGBM is evaluated in the later tuning step.

In [96]:
from sklearn.metrics import mean_squared_error
import numpy as np

def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2)) * 100


def evaluate_model(model, X_train_model, y_train_model, X_test_model, y_test_model):
    model.fit(X_train_model, y_train_model)

    train_pred = model.predict(X_train_model)
    test_pred = model.predict(X_test_model)

    return {
        "train_rmspe": rmspe(y_train_model, train_pred),
        "test_rmspe": rmspe(y_test_model, test_pred),
        "train_mse": mean_squared_error(y_train_model, train_pred),
        "test_mse": mean_squared_error(y_test_model, test_pred),
        "train_rmse": np.sqrt(mean_squared_error(y_train_model, train_pred)),
        "test_rmse": np.sqrt(mean_squared_error(y_test_model, test_pred)),
    }


results = []
results.append(
    (
        "Linear Regression",
        evaluate_model(
            LinearRegression(),
            X_train_scaled,
            y_train,
            X_test_scaled,
            y_test,
        ),
    )
)
results.append(
    (
        "KNN",
        evaluate_model(
            KNeighborsRegressor(n_neighbors=10),
            X_train_scaled,
            y_train,
            X_test_scaled,
            y_test,
        ),
    )
)
results.append(
    (
        "Decision Tree",
        evaluate_model(
            DecisionTreeRegressor(max_depth=6, random_state=12),
            X_train_scaled,
            y_train,
            X_test_scaled,
            y_test,
        ),
    )
)

for name, score in results:
    print(
        f"{name} - "
        f"Train RMSPE: {score['train_rmspe']:.2f}% | "
        f"Test RMSPE: {score['test_rmspe']:.2f}% | "
        f"Train MSE: {score['train_mse']:.2f} | "
        f"Test MSE: {score['test_mse']:.2f} | "
        f"Train RMSE: {score['train_rmse']:.2f} | "
        f"Test RMSE: {score['test_rmse']:.2f}"
    )

Linear Regression - Train RMSPE: 26.68% | Test RMSPE: 26.70% | Train MSE: 7063516.05 | Test MSE: 7725866.48 | Train RMSE: 2657.73 | Test RMSE: 2779.54
KNN - Train RMSPE: 66.26% | Test RMSPE: 67.92% | Train MSE: 20657730.18 | Test MSE: 28636545.25 | Train RMSE: 4545.08 | Test RMSE: 5351.31
Decision Tree - Train RMSPE: 74.93% | Test RMSPE: 71.04% | Train MSE: 15434563.67 | Test MSE: 23737632.69 | Train RMSE: 3928.68 | Test RMSE: 4872.13


In [100]:
from sklearn.metrics import mean_squared_error
import numpy as np

def mse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# Decision Tree
dtr_train_pred = best_dtr.predict(X_train_scaled)
dtr_test_pred = best_dtr.predict(X_test_scaled)

print("Decision Tree")
print(f"Train RMSPE: {rmspe(y_train, dtr_train_pred):.2f}%")
print(f"Test RMSPE: {rmspe(y_test, dtr_test_pred):.2f}%")
print(f"Train MSE: {mse(y_train, dtr_train_pred):.2f}")
print(f"Test MSE: {mse(y_test, dtr_test_pred):.2f}")
print(f"Train RMSE: {rmse(y_train, dtr_train_pred):.2f}")
print(f"Test RMSE: {rmse(y_test, dtr_test_pred):.2f}")


# KNN
knn_train_pred = best_knn.predict(X_train_scaled)
knn_test_pred = best_knn.predict(X_test_scaled)

print("\nKNN")
print(f"Train RMSPE: {rmspe(y_train, knn_train_pred):.2f}%")
print(f"Test RMSPE: {rmspe(y_test, knn_test_pred):.2f}%")
print(f"Train MSE: {mse(y_train, knn_train_pred):.2f}")
print(f"Test MSE: {mse(y_test, knn_test_pred):.2f}")
print(f"Train RMSE: {rmse(y_train, knn_train_pred):.2f}")
print(f"Test RMSE: {rmse(y_test, knn_test_pred):.2f}")


# LightGBM
lgbm_train_pred = best_lgbm.predict(X_train)
lgbm_test_pred = best_lgbm.predict(X_test)

print("\nLightGBM")
print(f"Train RMSPE: {rmspe(y_train, lgbm_train_pred):.2f}%")
print(f"Test RMSPE: {rmspe(y_test, lgbm_test_pred):.2f}%")
print(f"Train MSE: {mse(y_train, lgbm_train_pred):.2f}")
print(f"Test MSE: {mse(y_test, lgbm_test_pred):.2f}")
print(f"Train RMSE: {rmse(y_train, lgbm_train_pred):.2f}")
print(f"Test RMSE: {rmse(y_test, lgbm_test_pred):.2f}")

Decision Tree
Train RMSPE: 81.86%
Test RMSPE: 77.51%
Train MSE: 30405424.83
Test MSE: 31408310.10
Train RMSE: 5514.11
Test RMSE: 5604.31

KNN
Train RMSPE: 0.00%
Test RMSPE: 67.37%
Train MSE: 0.00
Test MSE: 26855827.61
Train RMSE: 0.00
Test RMSE: 5182.26

LightGBM
Train RMSPE: 26.34%
Test RMSPE: 28.58%
Train MSE: 7800489.16
Test MSE: 13787587.78
Train RMSE: 2792.94
Test RMSE: 3713.16


## 14. Model Tuning

The original GridSearchCV cells tune the existing Decision Tree, KNN, and LightGBM regressors using training data. The held-out test set is reserved for the final score reports.

In [98]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from lightgbm import LGBMRegressor
import numpy as np


def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2)) * 100


# Decision Tree Regressor GridSearch
dtr_params = {
    "max_depth": [4, 5, 6, 7, 8, 9, 10, 12, 15, 30],
    "max_features": [2, 3, 4, 5, 6, 7, 10, 12, 15],
    "max_leaf_nodes": [2, 3, 4, 5, 6, 7],
}

dtr_gridCV = GridSearchCV(
    DecisionTreeRegressor(random_state=12),
    param_grid=dtr_params,
    cv=5,
)
dtr_gridCV.fit(X_train_scaled, y_train)
print("Best DTR params:", dtr_gridCV.best_params_)
best_dtr = dtr_gridCV.best_estimator_

dtr_train_pred = best_dtr.predict(X_train_scaled)
dtr_test_pred = best_dtr.predict(X_test_scaled)
print(f"Best tuned DTR train RMSPE: {rmspe(y_train, dtr_train_pred):.2f}%")
print(f"Best tuned DTR test RMSPE: {rmspe(y_test, dtr_test_pred):.2f}%")


# KNN Regressor GridSearch
knn_params = {
    "n_neighbors": [5, 6, 7, 8, 9, 10, 12, 15, 20],
    "weights": ["uniform", "distance"],
    "p": [1, 2],
}

knn_gridCV = GridSearchCV(
    KNeighborsRegressor(),
    param_grid=knn_params,
    cv=5,
)
knn_gridCV.fit(X_train_scaled, y_train)
print("Best KNN params:", knn_gridCV.best_params_)
best_knn = knn_gridCV.best_estimator_

knn_train_pred = best_knn.predict(X_train_scaled)
knn_test_pred = best_knn.predict(X_test_scaled)
print(f"Best tuned KNN train RMSPE: {rmspe(y_train, knn_train_pred):.2f}%")
print(f"Best tuned KNN test RMSPE: {rmspe(y_test, knn_test_pred):.2f}%")


# LightGBM Regressor GridSearch
lgbm_params = {
    "n_estimators": [100, 200, 300, 400],
    "learning_rate": [0.01, 0.03, 0.05, 0.07, 0.1],
    "max_depth": [3, 5, 7, 9],
    "num_leaves": [15, 31, 50],
}

lgbm_gridCV = GridSearchCV(
    LGBMRegressor(random_state=12, verbosity=-1),
    param_grid=lgbm_params,
    cv=5,
)
lgbm_gridCV.fit(X_train, y_train)
print("Best LightGBM params:", lgbm_gridCV.best_params_)
best_lgbm = lgbm_gridCV.best_estimator_

lgbm_train_pred = best_lgbm.predict(X_train)
lgbm_test_pred = best_lgbm.predict(X_test)
print(f"Best tuned LightGBM train RMSPE: {rmspe(y_train, lgbm_train_pred):.2f}%")
print(f"Best tuned LightGBM test RMSPE: {rmspe(y_test, lgbm_test_pred):.2f}%")

Best DTR params: {'max_depth': 5, 'max_features': 12, 'max_leaf_nodes': 7}
Best tuned DTR train RMSPE: 81.86%
Best tuned DTR test RMSPE: 77.51%
Best KNN params: {'n_neighbors': 8, 'p': 2, 'weights': 'distance'}
Best tuned KNN train RMSPE: 0.00%
Best tuned KNN test RMSPE: 67.37%
Best LightGBM params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'num_leaves': 15}
Best tuned LightGBM train RMSPE: 26.34%
Best tuned LightGBM test RMSPE: 28.58%


## 15. Final Model Evaluation

The evaluation below includes the Linear Regression baseline and the tuned Decision Tree, KNN, and LightGBM models. Training and test RMSPE, MSE, and RMSE are reported.

In [101]:
from sklearn.metrics import mean_squared_error
import numpy as np

def mse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred)


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


# Linear Regression baseline
for model_name, score in results:
    if model_name == "Linear Regression":
        print("Linear Regression (baseline)")
        print(f"Train RMSPE: {score['train_rmspe']:.2f}%")
        print(f"Test RMSPE: {score['test_rmspe']:.2f}%")
        print(f"Train MSE: {score['train_mse']:.2f}")
        print(f"Test MSE: {score['test_mse']:.2f}")
        print(f"Train RMSE: {score['train_rmse']:.2f}")
        print(f"Test RMSE: {score['test_rmse']:.2f}")
        break


# Decision Tree

dtr_train_pred = best_dtr.predict(X_train_scaled)
dtr_test_pred = best_dtr.predict(X_test_scaled)

print("\nDecision Tree")
print(f"Train RMSPE: {rmspe(y_train, dtr_train_pred):.2f}%")
print(f"Test RMSPE: {rmspe(y_test, dtr_test_pred):.2f}%")
print(f"Train MSE: {mse(y_train, dtr_train_pred):.2f}")
print(f"Test MSE: {mse(y_test, dtr_test_pred):.2f}")
print(f"Train RMSE: {rmse(y_train, dtr_train_pred):.2f}")
print(f"Test RMSE: {rmse(y_test, dtr_test_pred):.2f}")


# KNN

knn_train_pred = best_knn.predict(X_train_scaled)
knn_test_pred = best_knn.predict(X_test_scaled)

print("\nKNN")
print(f"Train RMSPE: {rmspe(y_train, knn_train_pred):.2f}%")
print(f"Test RMSPE: {rmspe(y_test, knn_test_pred):.2f}%")
print(f"Train MSE: {mse(y_train, knn_train_pred):.2f}")
print(f"Test MSE: {mse(y_test, knn_test_pred):.2f}")
print(f"Train RMSE: {rmse(y_train, knn_train_pred):.2f}")
print(f"Test RMSE: {rmse(y_test, knn_test_pred):.2f}")


# LightGBM

lgbm_train_pred = best_lgbm.predict(X_train)
lgbm_test_pred = best_lgbm.predict(X_test)

print("\nLightGBM")
print(f"Train RMSPE: {rmspe(y_train, lgbm_train_pred):.2f}%")
print(f"Test RMSPE: {rmspe(y_test, lgbm_test_pred):.2f}%")
print(f"Train MSE: {mse(y_train, lgbm_train_pred):.2f}")
print(f"Test MSE: {mse(y_test, lgbm_test_pred):.2f}")
print(f"Train RMSE: {rmse(y_train, lgbm_train_pred):.2f}")
print(f"Test RMSE: {rmse(y_test, lgbm_test_pred):.2f}")

Linear Regression (baseline)
Train RMSPE: 26.68%
Test RMSPE: 26.70%
Train MSE: 7063516.05
Test MSE: 7725866.48
Train RMSE: 2657.73
Test RMSE: 2779.54

Decision Tree
Train RMSPE: 81.86%
Test RMSPE: 77.51%
Train MSE: 30405424.83
Test MSE: 31408310.10
Train RMSE: 5514.11
Test RMSE: 5604.31

KNN
Train RMSPE: 0.00%
Test RMSPE: 67.37%
Train MSE: 0.00
Test MSE: 26855827.61
Train RMSE: 0.00
Test RMSE: 5182.26

LightGBM
Train RMSPE: 26.34%
Test RMSPE: 28.58%
Train MSE: 7800489.16
Test MSE: 13787587.78
Train RMSE: 2792.94
Test RMSE: 3713.16


## 16. Key Findings

The evaluation cells above print the training and test metrics. Compare the train and test errors when assessing generalization; this notebook does not calculate feature importance or causal effects.

## 17. Conclusion

This project predicts customer credit card consumption (`cc_cons`) using the existing customer consumption, behavioral, and demographic fields. It demonstrates data inspection, manual missing-value handling, categorical encoding, scaling, regression modeling, and model evaluation.

## 18. Limitations and Future Improvements

The source files are not included in the repository, and the available column documentation and target time horizon are limited. Results are based on one random train-test split and may not represent future customers. Confirm that the monthly features precede the target period before using the model for future prediction.

Possible future work includes documenting the source and prediction horizon, testing with an out-of-time split, broader cross-validation and tuning, and assessing explainability and deployment monitoring.

## 19. GitHub Readiness

Data loading uses a repository-relative path. The Excel files are excluded from version control and should only be shared when licensing and privacy permit. Run all code cells from top to bottom with authorized data files present to reproduce the analysis.